# 30 - GridSearchCV, metricas de clasificacion y validacion cruzada K-fold

Entrenar un modelo es solo una parte del trabajo. Tambien debemos responder tres preguntas:

1. ¿Que significa que el modelo sea bueno para este problema?
2. ¿Como estimamos su rendimiento en datos que no ha visto?
3. ¿Como elegimos hiperparametros sin contaminar la evaluacion final?

Esta sesion conecta esas preguntas mediante matrices de confusion, precision, recall, F1-score, validacion cruzada y GridSearchCV.

> Nota de vocabulario: el nombre habitual es **K-fold cross-validation** o **validacion cruzada K-fold**. A veces se dice informalmente cross-fold validation, pero no es el termino estandar.

## Objetivos de aprendizaje

Al terminar podras:

- interpretar TP, TN, FP y FN desde el significado real de las clases;
- explicar cuando accuracy, precision, recall o F1-score pueden ser engañosos;
- usar KFold y StratifiedKFold con criterio;
- comparar modelos con los mismos folds y reportar media y variabilidad;
- construir un Pipeline que evite fuga de informacion;
- configurar GridSearchCV con una o varias metricas;
- reservar el conjunto de prueba para una evaluacion final honesta;
- distinguir seleccion de hiperparametros de seleccion del umbral.

## Ruta de la sesion

1. Separar entrenamiento, validacion y prueba.
2. Leer la matriz de confusion.
3. Entender accuracy, precision, recall, especificidad y F1.
4. Descubrir la trampa del desbalance de clases.
5. Aplicar validacion cruzada K-fold y estratificacion.
6. Evaluar varias metricas con cross_validate.
7. Buscar hiperparametros con GridSearchCV.
8. Evaluar una sola vez en test.
9. Estudiar el umbral de decision.
10. Resolver ejercicios que exigen justificar decisiones.

## 1) Preparacion

Usaremos datos sinteticos para aislar conceptos y el conjunto de cancer de mama incluido en scikit-learn para integrar el flujo completo. No se descargan datos externos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 30

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

## 2) Entrenamiento, validacion y prueba cumplen funciones distintas

Un flujo honesto separa responsabilidades:

| Parte | Para que se usa | ¿Puede influir en decisiones? |
|---|---|---|
| Train | Ajustar parametros internos del modelo | Si |
| Validacion | Comparar modelos, hiperparametros o umbrales | Si |
| Test | Estimar el rendimiento final | No |

Con validacion cruzada, train se divide temporalmente en varios pares de entrenamiento-validacion. El test externo permanece guardado.

Si miramos test para elegir C, una metrica, variables o un umbral, test deja de representar datos realmente nuevos. No importa que nunca llamemos fit con test: usar su resultado para decidir tambien filtra informacion.

## 3) Un problema desbalanceado

Crearemos una clasificacion binaria donde aproximadamente 8 % de los casos pertenecen a la clase positiva. Imagina que 1 significa una operacion posiblemente fraudulenta y 0 una operacion normal.

El significado de la clase positiva no es un detalle: determina como interpretamos precision y recall.

In [ ]:
X_imb, y_imb = make_classification(
    n_samples=1600,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    weights=[0.92, 0.08],
    class_sep=1.0,
    flip_y=0.02,
    random_state=RANDOM_STATE,
)

X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imb,
    y_imb,
    test_size=0.25,
    stratify=y_imb,
    random_state=RANDOM_STATE,
)

distribution = pd.DataFrame({
    "conjunto": ["completo", "train", "test"],
    "n": [len(y_imb), len(y_train_imb), len(y_test_imb)],
    "proporcion_positiva": [y_imb.mean(), y_train_imb.mean(), y_test_imb.mean()],
})
distribution.round(4)

### 3.1 La trampa de accuracy

Un clasificador que siempre responde 0 acertara cerca de 92 % de las veces. Sin embargo, no detectara ningun caso positivo.

$$
Accuracy = \frac{predicciones\ correctas}{total\ de\ predicciones}
$$

Accuracy responde cuantos casos se acertaron, pero no distingue que tipo de error se cometio. Cuando las clases estan desbalanceadas o los errores tienen costos diferentes, debemos abrir la matriz de confusion.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_imb, y_train_imb)
y_dummy = dummy.predict(X_test_imb)

print("Accuracy del modelo que siempre predice 0:", round(accuracy_score(y_test_imb, y_dummy), 4))
print("Recall de la clase positiva:", recall_score(y_test_imb, y_dummy, zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test_imb,
    y_dummy,
    display_labels=["normal (0)", "positivo (1)"],
    cmap="Blues",
)
plt.title("Accuracy alto, utilidad nula para detectar positivos")
plt.show()

## 4) Matriz de confusion y metricas

Para una clase positiva 1:

| | Predice 0 | Predice 1 |
|---|---:|---:|
| Real 0 | TN: verdadero negativo | FP: falso positivo |
| Real 1 | FN: falso negativo | TP: verdadero positivo |

A partir de estas cuatro cantidades:

$$
Precision = \frac{TP}{TP+FP}
\qquad
Recall = \frac{TP}{TP+FN}
$$

$$
Especificidad = \frac{TN}{TN+FP}
\qquad
F1 = 2\frac{Precision \cdot Recall}{Precision+Recall}
$$

- **Precision**: de todo lo marcado como positivo, ¿cuanto era realmente positivo?
- **Recall o sensibilidad**: de todos los positivos reales, ¿cuantos encontramos?
- **Especificidad**: de todos los negativos reales, ¿cuantos descartamos correctamente?
- **F1-score**: media armonica de precision y recall; solo es alto si ambas son razonables.

F1 no usa TN. Por eso tampoco es una metrica universal.

### 4.1 Una funcion para no ocultar ninguna pieza

La funcion siguiente devuelve las metricas y tambien los cuatro conteos. En un reporte real conviene mostrar ambos: una sola cifra puede ocultar el tipo y la magnitud de los errores.

In [ ]:
def binary_metrics(y_true, y_pred, y_score=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    row = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if tn + fp else np.nan,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }
    if y_score is not None:
        row["roc_auc"] = roc_auc_score(y_true, y_score)
    return row


logistic_imb = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
])
logistic_imb.fit(X_train_imb, y_train_imb)

y_logistic = logistic_imb.predict(X_test_imb)
score_logistic = logistic_imb.predict_proba(X_test_imb)[:, 1]

comparison_holdout = pd.DataFrame(
    [
        binary_metrics(y_test_imb, y_dummy),
        binary_metrics(y_test_imb, y_logistic, score_logistic),
    ],
    index=["siempre clase 0", "regresion logistica"],
)
comparison_holdout.round(4)

### 4.2 No existe una mejor metrica sin contexto

- Si un falso positivo activa una revision manual costosa, precision puede importar mucho.
- Si un falso negativo deja pasar una enfermedad o fraude grave, recall puede ser prioritario.
- Si queremos equilibrio entre precision y recall, F1 es una opcion, pero asigna a ambas la misma importancia.
- Si los errores son aproximadamente simetricos y las clases estan equilibradas, accuracy puede ser suficiente.
- ROC-AUC evalua ordenamiento a traves de muchos umbrales; no describe por si solo la operacion en un umbral concreto.

Primero se define el costo del error; despues se elige la metrica. Elegir la metrica que produce el numero mas alto invierte el razonamiento.

## 5) Validacion cruzada K-fold

K-fold divide los datos de desarrollo en K partes:

1. entrena con K - 1 folds y valida con el restante;
2. repite hasta que cada fold haya sido validacion una vez;
3. resume las K puntuaciones mediante su media y dispersion.

Con K = 5, cada modelo usa aproximadamente 80 % para ajustar y 20 % para validar en cada ronda.

K-fold aprovecha mejor los datos que una sola division, pero no crea datos nuevos. Los resultados de los folds comparten observaciones de entrenamiento, de modo que no son cinco experimentos totalmente independientes.

In [ ]:
def show_folds(n_samples=20, n_splits=5):
    cv = KFold(n_splits=n_splits, shuffle=False)
    grid = np.full((n_splits, n_samples), "train", dtype=object)
    for fold, (_, valid_idx) in enumerate(cv.split(np.arange(n_samples))):
        grid[fold, valid_idx] = "validacion"
    return pd.DataFrame(grid, index=[f"ronda {i + 1}" for i in range(n_splits)])


show_folds()

### 5.1 KFold frente a StratifiedKFold

En clasificacion, StratifiedKFold intenta conservar en cada fold la proporcion de cada clase. Esto evita que un fold reciba accidentalmente muy pocos positivos.

Estratificar no resuelve todos los problemas:

- si varias filas pertenecen a la misma persona, se requiere una estrategia por grupos;
- si hay orden temporal, mezclar futuro y pasado es incorrecto;
- si los datos duplicados aparecen en folds distintos, la estimacion sera optimista.

La estrategia de division debe representar como llegaran los datos nuevos.

In [ ]:
y_ordered = np.sort(y_train_imb)
plain_cv = KFold(n_splits=5, shuffle=False)
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, cv in [("KFold sin mezclar", plain_cv), ("StratifiedKFold", stratified_cv)]:
    for fold, (_, valid_idx) in enumerate(cv.split(np.zeros(len(y_ordered)), y_ordered), start=1):
        rows.append({
            "estrategia": name,
            "fold": fold,
            "n_validacion": len(valid_idx),
            "positivos_validacion": int(y_ordered[valid_idx].sum()),
            "proporcion_positiva": y_ordered[valid_idx].mean(),
        })

pd.DataFrame(rows).round(4)

## 6) Preprocesamiento dentro de Pipeline

En cada fold, cualquier transformacion que aprenda de los datos debe ajustarse solo con la parte de entrenamiento de esa ronda. StandardScaler aprende medias y desviaciones; imputadores, selectores de variables y PCA tambien aprenden informacion.

Pipeline garantiza esta secuencia dentro de cada fold:

$$
train\ del\ fold \rightarrow ajustar\ scaler \rightarrow ajustar\ modelo
$$

$$
validacion\ del\ fold \rightarrow transformar\ con\ scaler\ ya\ ajustado \rightarrow evaluar
$$

Escalar todo el dataset antes de cross-validation produce fuga de informacion, aunque las etiquetas no entren al escalador.

## 7) Evaluar varias metricas con cross_validate

cross_val_score devuelve normalmente una metrica. cross_validate permite varias metricas, tiempos y, si se solicita, puntuaciones de entrenamiento.

Para comparar modelos justamente usaremos el mismo objeto StratifiedKFold. Reportaremos media y desviacion entre folds; la media sola oculta inestabilidad.

In [ ]:
cv_imb = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

models = {
    "dummy": DummyClassifier(strategy="prior"),
    "logistica": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]),
}

cv_results = {}
for name, model in models.items():
    cv_results[name] = cross_validate(
        model,
        X_train_imb,
        y_train_imb,
        cv=cv_imb,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
    )

summary_rows = []
for model_name, result in cv_results.items():
    for metric in scoring:
        values = result[f"test_{metric}"]
        summary_rows.append({
            "modelo": model_name,
            "metrica": metric,
            "media_validacion": values.mean(),
            "desviacion_folds": values.std(ddof=1),
            "media_train": result[f"train_{metric}"].mean(),
        })

cv_summary = pd.DataFrame(summary_rows)
cv_summary.round(4)

### 7.1 Como leer los resultados

- Una media de validacion mayor es mejor solo para la metrica que el problema realmente necesita.
- Una dispersion grande indica sensibilidad a la muestra; puede requerir mas datos, folds repetidos o investigar subgrupos.
- Una brecha amplia entre train y validacion es una señal de posible sobreajuste, no una prueba automatica.
- Un modelo base como DummyClassifier revela si el modelo aprendido aporta algo sobre una regla trivial.

La desviacion entre folds describe variacion de esta particion. No debe presentarse automaticamente como un intervalo de confianza estadistico.

## 8) Que hace GridSearchCV

Un parametro se aprende durante fit. Un **hiperparametro** se fija antes de entrenar: por ejemplo C, profundidad maxima o numero de vecinos.

GridSearchCV:

1. enumera todas las combinaciones indicadas en param_grid;
2. evalua cada combinacion con validacion cruzada;
3. ordena candidatos usando scoring;
4. con refit, vuelve a ajustar la combinacion elegida usando todo train.

Si hay 10 combinaciones y 5 folds, se realizan 50 ajustes durante la busqueda, mas el ajuste final. Grid search no descubre valores fuera de la cuadricula y una cuadricula enorme puede ser costosa.

### 8.1 Decisiones antes de ejecutar la busqueda

Antes del codigo deben quedar claras estas decisiones:

- metrica principal y justificacion;
- estrategia de folds;
- rango razonable de hiperparametros;
- transformaciones que deben vivir en Pipeline;
- test externo que no participara en la seleccion.

Cuando scoring contiene varias metricas, refit indica cual decide best_estimator_. Asi podemos inspeccionar varios criterios sin fingir que todos producen la misma eleccion.

## 9) Caso completo: detectar tumores malignos

El dataset original codifica maligno como 0 y benigno como 1. Las metricas binarias de scikit-learn consideran positiva la clase 1 por defecto.

Para que recall responda cuantos tumores malignos detectamos, recodificaremos:

- 1 = maligno;
- 0 = no maligno.

No hacer explicita esta decision puede producir una metrica matematicamente correcta pero semanticamente equivocada.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X_cancer = cancer.data
y_cancer = (cancer.target == 0).astype(int).rename("maligno")

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cancer,
    y_cancer,
    test_size=0.25,
    stratify=y_cancer,
    random_state=RANDOM_STATE,
)

print("Train:", X_train_c.shape, "| Test:", X_test_c.shape)
print("Proporcion maligna en train:", round(y_train_c.mean(), 4))
print("Proporcion maligna en test:", round(y_test_c.mean(), 4))

In [ ]:
cancer_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"],
}

cv_grid = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring_grid = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

grid = GridSearchCV(
    estimator=cancer_pipeline,
    param_grid=param_grid,
    scoring=scoring_grid,
    refit="f1",
    cv=cv_grid,
    return_train_score=True,
    n_jobs=1,
)
grid.fit(X_train_c, y_train_c)

print("Candidatos:", len(grid.cv_results_["params"]))
print("Mejores hiperparametros segun F1:", grid.best_params_)
print("F1 medio de validacion del elegido:", round(grid.best_score_, 4))

### 9.1 Inspeccionar la busqueda completa

best_score_ no basta. cv_results_ permite estudiar compromisos: un candidato puede ganar en F1 y otro en recall. Tambien muestra variabilidad y diferencias entre train y validacion.

In [ ]:
grid_table = pd.DataFrame(grid.cv_results_)
columns = [
    "param_model__C",
    "param_model__class_weight",
    "mean_train_f1",
    "mean_test_accuracy",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_f1",
    "std_test_f1",
    "mean_test_roc_auc",
    "rank_test_f1",
]

grid_table[columns].sort_values("rank_test_f1").reset_index(drop=True).round(4)

El primer lugar no siempre domina en todo. Si el requisito real fuera recall minimo de 0.98, refit igual a F1 podria ser una especificacion incorrecta. La seleccion debe reflejar una regla definida antes de mirar test.

Diferencias minimas entre candidatos no garantizan una superioridad practica. Conviene considerar dispersion, complejidad, estabilidad y costo operativo.

## 10) Evaluacion final en test

GridSearchCV ya reajusto la mejor combinacion sobre todo train porque usamos refit. Ahora usamos test para estimar el rendimiento final.

Esta evaluacion no debe iniciar una nueva ronda de cambios. Si modificamos el sistema despues de verla, necesitaremos reconocer que test ya participo o conseguir una evaluacion verdaderamente nueva.

In [ ]:
best_model = grid.best_estimator_
y_pred_c = best_model.predict(X_test_c)
y_score_c = best_model.predict_proba(X_test_c)[:, 1]

final_metrics = pd.Series(
    binary_metrics(y_test_c, y_pred_c, y_score_c),
    name="test_final",
)
display(final_metrics.round(4))

print(classification_report(
    y_test_c,
    y_pred_c,
    target_names=["no maligno", "maligno"],
    digits=3,
))

ConfusionMatrixDisplay.from_predictions(
    y_test_c,
    y_pred_c,
    display_labels=["no maligno", "maligno"],
    cmap="Blues",
)
plt.title("Evaluacion final con umbral 0.5")
plt.show()

## 11) El umbral cambia precision y recall

predict_proba produce una puntuacion entre 0 y 1. predict suele convertirla en clase positiva cuando alcanza 0.5.

- bajar el umbral suele detectar mas positivos: sube recall y pueden aparecer mas falsos positivos;
- subir el umbral exige mas evidencia: suele subir precision y pueden aparecer mas falsos negativos.

El umbral 0.5 no es una ley natural. Debe elegirse con datos de validacion y una funcion de costo o restriccion explicita, nunca probando umbrales sobre el test final.

### 11.1 Predicciones fuera de fold para estudiar el umbral

cross_val_predict genera para cada fila de train una prediccion hecha por un modelo que no entreno con esa fila. Asi podemos explorar umbrales sin tocar test.

Los hiperparametros ya fueron elegidos con train, por lo que un analisis de produccion mas estricto separaria otra validacion o anidaria todas las decisiones. Aqui mantenemos test aislado y hacemos visible la limitacion.

In [ ]:
oof_scores = cross_val_predict(
    clone(best_model),
    X_train_c,
    y_train_c,
    cv=cv_grid,
    method="predict_proba",
    n_jobs=1,
)[:, 1]

threshold_rows = []
for threshold in np.arange(0.10, 0.91, 0.05):
    pred = (oof_scores >= threshold).astype(int)
    metrics = binary_metrics(y_train_c, pred)
    threshold_rows.append({"threshold": threshold, **metrics})

threshold_table = pd.DataFrame(threshold_rows)
threshold_table[["threshold", "precision", "recall", "specificity", "f1", "fp", "fn"]].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
PrecisionRecallDisplay.from_predictions(
    y_train_c,
    oof_scores,
    name="predicciones fuera de fold",
    ax=ax,
)
ax.set_title("Compromiso precision-recall en datos de desarrollo")
plt.show()

# Ejemplo de regla previa: recall de al menos 0.95 y, entre los
# umbrales factibles, elegir el de mayor precision.
feasible = threshold_table[threshold_table["recall"] >= 0.95]
if feasible.empty:
    selected_threshold = 0.5
    print("Ningun umbral evaluado satisface el recall requerido.")
else:
    selected_threshold = feasible.sort_values(
        ["precision", "threshold"], ascending=[False, False]
    ).iloc[0]["threshold"]
    print("Umbral elegido sin usar test:", round(float(selected_threshold), 2))

In [ ]:
pred_default = (y_score_c >= 0.5).astype(int)
pred_selected = (y_score_c >= selected_threshold).astype(int)

threshold_comparison_test = pd.DataFrame(
    [
        binary_metrics(y_test_c, pred_default, y_score_c),
        binary_metrics(y_test_c, pred_selected, y_score_c),
    ],
    index=["umbral 0.50", f"umbral {selected_threshold:.2f}"],
)
threshold_comparison_test.round(4)

## 12) Guia breve para elegir una metrica

| Necesidad principal | Metrica candidata | Pregunta que responde |
|---|---|---|
| Errores simetricos y clases equilibradas | Accuracy | ¿Que proporcion total acerte? |
| Evitar falsas alarmas | Precision | ¿Que tan confiable es una alerta positiva? |
| No perder positivos | Recall | ¿Que proporcion de positivos encontre? |
| Equilibrar precision y recall | F1 | ¿Ambas son razonables en una sola cifra? |
| Descartar negativos correctamente | Especificidad | ¿Que proporcion de negativos reconoci? |
| Evaluar ranking entre clases | ROC-AUC | ¿Los positivos suelen recibir mayor score? |
| Ranking con positivos escasos | Average precision / curva PR | ¿Como se comporta precision al recuperar positivos? |

Una aplicacion puede necesitar restricciones, no solo maximizar una cifra: por ejemplo, maximizar precision sujeto a recall mayor o igual que 0.95.

## 13) F1 en problemas multiclase

En multiclase se calcula una metrica por clase tratandola temporalmente como positiva:

- **macro**: promedio simple; cada clase pesa igual, aunque sea pequeña;
- **weighted**: promedio ponderado por soporte; las clases grandes dominan;
- **micro**: suma TP, FP y FN globales antes de calcular; en clasificacion multiclase de una sola etiqueta, precision micro, recall micro y F1 micro coinciden con accuracy;
- **per-class**: no promedia; permite localizar la clase problematica.

F1 weighted puede verse alto aunque una clase minoritaria funcione mal. Por eso conviene acompañarlo con F1 macro, matriz de confusion y metricas por clase.

## 14) Errores metodologicos frecuentes

1. Elegir la metrica despues de ver cual favorece al modelo.
2. Usar accuracy con clases desbalanceadas sin inspeccionar errores.
3. Olvidar que clase es positiva.
4. Escalar, imputar o seleccionar variables antes de los folds.
5. Ajustar hiperparametros en test.
6. Comparar modelos con particiones diferentes y atribuir toda diferencia al algoritmo.
7. Reportar solo la mejor media sin dispersion ni baseline.
8. Interpretar best_score_ como rendimiento final de test.
9. Probar una cuadricula enorme sin una hipotesis sobre rangos.
10. Tratar filas correlacionadas, grupos o tiempo como observaciones independientes.

## 15) Extension: validacion cruzada anidada

La misma validacion usada para escoger entre muchos candidatos puede producir una estimacion optimista del ganador. La validacion cruzada anidada separa dos niveles:

- ciclo interno: selecciona hiperparametros;
- ciclo externo: estima el procedimiento completo de seleccion.

Es util al comparar procedimientos con pocos datos o muchas decisiones. Si existe un test externo bien protegido, este ya proporciona una evaluacion final; la validacion anidada puede aportar una estimacion mas robusta durante desarrollo, aunque cuesta muchos mas ajustes.

## 16) Checklist de aprendizaje

Intenta responder sin ejecutar codigo:

1. ¿Por que un modelo con 95 % de accuracy puede ser inutil?
2. ¿Que denominador distingue precision de recall?
3. ¿Por que F1 no resume el desempeño sobre verdaderos negativos?
4. ¿Que problema reduce StratifiedKFold?
5. ¿Por que StandardScaler debe estar dentro de Pipeline?
6. ¿Que significa refit igual a f1 en GridSearchCV?
7. ¿Por que best_score_ no sustituye la evaluacion en test?
8. ¿Como afecta bajar el umbral a FP y FN?
9. ¿Cuando K-fold aleatorio seria una mala representacion del futuro?
10. ¿Que diferencia hay entre optimizar hiperparametros y elegir un umbral?

## 17) Ejercicios para pensar y experimentar

Para cada ejercicio sigue este protocolo:

1. escribe una hipotesis antes de programar;
2. define que resultado refutaria tu hipotesis;
3. ejecuta el experimento;
4. presenta evidencia con una tabla o grafica pertinente;
5. redacta una conclusion que separe resultado numerico y decision practica.

No se evalua reproducir el codigo anterior, sino justificar el diseño.

### Ejercicio 1: reconstruir metricas a mano

Un sistema produjo TN = 940, FP = 40, FN = 15 y TP = 5.

1. Calcula accuracy, precision, recall, especificidad y F1 sin sklearn.
2. ¿Por que accuracy cuenta una historia diferente de recall?
3. Si la clase positiva representa una enfermedad tratable, ¿recomendarias el sistema?
4. Cambia exactamente uno de los cuatro conteos para representar una mejora de recall sin cambiar el total de casos. Explica que otro conteo debe cambiar tambien.
5. Comprueba al final tus calculos con una funcion de sklearn.

In [ ]:
# Ejercicio 1
tn, fp, fn, tp = 940, 40, 15, 5

# Escribe primero las formulas y calcula sin llamar metricas de sklearn.
# Despues construye y_true e y_pred para comprobar el resultado.

### Ejercicio 2: cuando accuracy premia no hacer nada

Usa X_train_imb y y_train_imb.

1. Entrena DummyClassifier, regresion logistica y regresion logistica con class_weight igual a balanced.
2. Evalua los tres con exactamente los mismos folds.
3. Antes de ejecutar, predice cual ganara en accuracy, recall y F1.
4. Busca un caso donde mejorar recall tenga un costo en precision.
5. Elige un modelo solo despues de inventar un costo razonable para FP y FN.

Tu conclusion debe explicar por que cambiar class_weight no es una mejora gratuita.

In [ ]:
# Ejercicio 2
candidate_models = {
    "dummy": DummyClassifier(strategy="prior"),
    "logistica": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000)),
    ]),
    # Agrega aqui la variante con class_weight.
}

# Reutiliza cv_imb y scoring. No uses test para elegir.

### Ejercicio 3: K controla un compromiso

Compara K = 3, 5, 10 y 20 con el mismo modelo y datos.

1. Predice como cambian el costo computacional y el tamaño del fold de validacion.
2. Registra media, desviacion y tiempo de ajuste para F1.
3. Repite todo con tres random_state diferentes.
4. ¿El K con mayor media es estable entre repeticiones?
5. ¿Elegirias K usando el mismo criterio con 200 filas y con dos millones?

No concluyas que un K es universalmente mejor: relaciona la eleccion con tamaño, costo y variabilidad.

In [ ]:
# Ejercicio 3
k_values = [3, 5, 10, 20]
seeds = [10, 20, 30]

# Construye un StratifiedKFold por cada combinacion de K y semilla.
# Guarda tambien fit_time devuelto por cross_validate.

### Ejercicio 4: la metrica cambia al ganador de GridSearchCV

Repite la busqueda del caso de cancer tres veces con refit igual a precision, recall y F1.

1. Antes de ejecutar, predice como cambiara class_weight.
2. Compara hiperparametros elegidos y metricas de validacion.
3. No evalues las tres alternativas repetidamente sobre test.
4. Escribe tres escenarios operativos donde cada eleccion seria defendible.
5. Diseña una regla con recall minimo en vez de maximizar una sola media.

Pregunta central: ¿GridSearchCV encontro el mejor modelo o solo el mejor segun la definicion que le diste?

In [ ]:
# Ejercicio 4
refit_metrics = ["precision", "recall", "f1"]

# Ajusta cada busqueda solo con X_train_c, y_train_c.
# Compara best_params_ y las filas correspondientes de cv_results_.
# Deja X_test_c, y_test_c fuera de esta exploracion.

### Ejercicio 5: demostrar fuga de informacion

Diseña dos flujos:

A. ajustar StandardScaler con todos los datos y despues hacer cross-validation;
B. colocar StandardScaler dentro de Pipeline.

1. Identifica exactamente que estadisticas del fold de validacion conoce A.
2. ¿Por que la diferencia numerica puede ser pequeña y aun asi A ser incorrecto?
3. Crea un dataset donde algunas filas tengan valores extremos e investiga si la diferencia aumenta.
4. Repite el razonamiento con imputacion de faltantes y seleccion de variables.

La meta no es fabricar una gran diferencia, sino explicar la violacion metodologica.

### Ejercicio 6: elegir umbral con costos

Usa y_train_c y oof_scores. Supone que:

- cada FN cuesta 20 unidades;
- cada FP cuesta 2 unidades;
- TP y TN tienen costo 0.

1. Calcula el costo total para umbrales de 0.05 a 0.95.
2. Elige el umbral de menor costo sin usar test.
3. Cambia la relacion de costos y explica por que cambia el umbral.
4. ¿Que supuestos debiles contiene sumar costos fijos por error?
5. ¿Que harias si las probabilidades estuvieran mal calibradas?

In [ ]:
# Ejercicio 6
cost_fn = 20
cost_fp = 2

# Para cada umbral recupera fp y fn con confusion_matrix.
# costo_total = cost_fn * fn + cost_fp * fp
# Grafica costo_total frente al umbral y argumenta la eleccion.

### Ejercicio 7: macro, weighted y una clase olvidada

Construye manualmente un problema de tres clases donde:

- la clase A tenga 900 ejemplos y se prediga muy bien;
- la clase B tenga 90 y se prediga de forma razonable;
- la clase C tenga 10 y nunca se prediga correctamente.

1. Calcula F1 macro, weighted y por clase.
2. ¿Que promedio hace mas visible el fracaso de C?
3. ¿Seria correcto afirmar que macro siempre es mejor?
4. Inventa dos aplicaciones donde el mismo resultado lleve a decisiones diferentes.

### Ejercicio 8: diseñar una evaluacion antes de modelar

Elige uno de estos problemas: fraude, abandono de clientes, spam o falla de maquinaria.

Sin entrenar nada, redacta un protocolo de una pagina que incluya:

1. unidad de observacion y significado de la clase positiva;
2. costo relativo de FP y FN;
3. metrica primaria y metricas de control;
4. division adecuada: aleatoria, estratificada, temporal o por grupos;
5. baseline minimo;
6. hiperparametros que vale la pena buscar y por que;
7. criterio para elegir umbral;
8. condiciones bajo las cuales rechazarias desplegar el modelo.

Compara tu protocolo con el de otra persona. Las diferencias de supuestos son parte del ejercicio.

## 18) Cierre

Una evaluacion confiable no empieza con GridSearchCV. Empieza definiendo que error importa y como deben simularse los datos futuros.

La secuencia completa es:

1. reservar un test externo;
2. elegir una estrategia de validacion coherente con el problema;
3. colocar preprocesamiento y modelo dentro de Pipeline;
4. comparar contra un baseline con metricas justificadas;
5. buscar hiperparametros solo en datos de desarrollo;
6. estudiar dispersion y compromisos, no solo el primer lugar;
7. fijar, si hace falta, un umbral mediante validacion;
8. evaluar una vez en test y reportar tambien los errores concretos.

GridSearchCV automatiza ajustes. No automatiza el significado del problema, el costo de los errores ni la honestidad del diseño experimental.